In [2]:
import pandas as pd

d   = pd.read_csv('./radio_features_clean_ref.csv')
raw = pd.read_csv('./radio_features.csv')      # <- le fichier qui porte coords + sessions

# Que contient réellement le fichier brut ?
print([c for c in raw.columns if c not in d.columns])

SESS = [c for c in ['Session_date', 'Session_time'] if c in raw.columns]
GEO  = [c for c in ['Rx_lat', 'Rx_lon'] if c in raw.columns]

if not SESS:
    print("\nPas de colonne de session dans radio_features.csv -> "
          "le contrôle est impossible, garde les deux limites séparées.")
else:
    d = d.join(raw.set_index('Ref')[SESS + GEO], on='Ref')

    # 1. Étendue spatiale de chaque session (~111 km par degré de latitude)
    g = d.groupby(SESS)
    print(g.agg(n=('Ref', 'size'),
                km_lat=('Rx_lat', lambda s: (s.max()-s.min())*111),
                km_lon=('Rx_lon', lambda s: (s.max()-s.min())*111*0.995))
           .sort_values('n', ascending=False).round(2))

    # 2. Deux sessions couvrent-elles la même maille ?
    d['cell'] = d.Rx_lat.round(3).astype(str) + '_' + d.Rx_lon.round(3).astype(str)
    chev = d.groupby('cell')[SESS[0]].nunique()
    print(f"\nMailles visitées par plusieurs sessions : "
          f"{(chev > 1).sum()} / {len(chev)}")

['Rx_lat', 'Rx_lon', 'Rx_power', 'FSM_spread', 'Prcp', 'Wpgt', 'Session_date', 'Session_time', 'City', 'Tx_id', 'Tx_lat', 'Tx_lon', 'Tx_alt', 'Tx_id_2nd', 'Freq_2nd', 'tx_ratio', 'Rx_alt', 'Slope_Tx_Rx', 'Roughness_Tx_Rx', 'Slope_Local_Rx_50m', 'Roughness_Local_Rx_50m']
                           n  km_lat  km_lon
Session_date Session_time                   
2023-12-13   11:06:27      1     0.0     0.0
2024-01-22   16:19:12      1     0.0     0.0
2024-11-14   12:10:48      1     0.0     0.0
             11:33:00      1     0.0     0.0
             11:13:01      1     0.0     0.0
...                       ..     ...     ...
2024-01-11   11:46:44      1     0.0     0.0
             11:28:11      1     0.0     0.0
2024-01-09   18:13:28      1     0.0     0.0
             17:55:16      1     0.0     0.0
2024-11-16   18:52:11      1     0.0     0.0

[324 rows x 3 columns]

Mailles visitées par plusieurs sessions : 1 / 323


In [3]:
import pandas as pd

d   = pd.read_csv('./radio_features_clean_ref.csv')
raw = pd.read_csv('./radio_features.csv')
d = d.join(raw.set_index('Ref')[['Session_date','Session_time','Rx_lat','Rx_lon','City']], on='Ref')

d['heure'] = pd.to_datetime(d.Session_time, format='mixed').dt.hour
d['bloc_h'] = d.Session_date + ' ' + d.heure.astype(str) + 'h'

for nom, cle in [('JOUR', 'Session_date'), ('HEURE', 'bloc_h')]:
    g = d.groupby(cle)
    r = g.agg(n=('Ref','size'),
              villes=('City','nunique'),
              km_lat=('Rx_lat', lambda s: (s.max()-s.min())*111),
              km_lon=('Rx_lon', lambda s: (s.max()-s.min())*111*0.995)).round(2)
    print(f"\n===== BLOC = {nom} : {len(r)} blocs, "
          f"médiane {r.n.median():.0f} points/bloc =====")
    print(r.sort_values('n', ascending=False).head(12))
    print(f"blocs à 1 seul point : {(r.n==1).sum()}/{len(r)}")
    print(f"étendue médiane : {r.km_lat.median():.1f} x {r.km_lon.median():.1f} km")

    # recouvrement : deux blocs sur la même maille de 500 m ?
    d['cell'] = (d.Rx_lat.round(3).astype(str) + '_' + d.Rx_lon.round(3).astype(str))
    ch = d.groupby('cell')[cle].nunique()
    print(f"mailles partagées par >1 bloc : {(ch>1).sum()}/{len(ch)}")


===== BLOC = JOUR : 17 blocs, médiane 18 points/bloc =====
               n  villes  km_lat  km_lon
Session_date                            
2024-11-16    40       1    2.34    3.94
2024-11-15    40       1    4.54    2.00
2024-11-14    36       1    1.87    2.73
2023-12-21    21       1    2.46    2.14
2024-01-12    20       1    1.19    2.64
2024-01-22    18       1    0.56    2.83
2024-01-16    18       1    2.38    3.12
2024-01-15    18       1    1.22    2.19
2024-01-11    18       1    1.32    1.90
2023-12-22    17       1    3.01    2.36
2023-12-19    17       1    1.24    1.30
2024-01-09    14       1    1.51    1.54
blocs à 1 seul point : 0/17
étendue médiane : 1.5 x 2.1 km
mailles partagées par >1 bloc : 1/323

===== BLOC = HEURE : 111 blocs, médiane 3 points/bloc =====
                  n  villes  km_lat  km_lon
bloc_h                                     
2024-11-16 9.0h   5       1    0.94    0.62
2024-01-11 16.0h  5       1    0.23    0.73
2024-11-16 8.0h   5       1    0